In [1]:

import copy
import torch
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import pytorch_lightning as pl
from pytorch_forecasting.data import GroupNormalizer
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_forecasting.data.examples import get_stallion_data
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters
warnings.filterwarnings("ignore")
from tqdm import tqdm
import matplotlib.pyplot as plt
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from datetime import timedelta


In [2]:
il_nosu=7


In [3]:
df_feat_model=pd.read_parquet("model_ready_data.parquet")
df_feat_model = df_feat_model.sort_values(["IlceKodu", "OrderDay"])

In [4]:
df_feat_model=pd.read_parquet("model_ready_data.parquet")
df_feat_model = df_feat_model.sort_values(["IlceKodu", "OrderDay"])

# Her IlceKodu için son 28 gün
test_df = (
    df_feat_model
    .groupby("IlceKodu", group_keys=False)
    .tail(105)
    .reset_index(drop=True)
)

In [5]:
base_df = test_df.copy()

In [6]:
if 'time_idx' not in base_df.columns:
    if base_df['OrderDay'].dtype != 'int64':
        base_df['OrderDay'] = pd.to_datetime(base_df['OrderDay'])
        base_df = base_df.sort_values(['IlceKodu', 'OrderDay'])
        base_df['time_idx'] = base_df.groupby('IlceKodu').cumcount()
    else:
        base_df['time_idx'] = base_df['OrderDay']

In [7]:
base_df

,OrderDay,IlKodu,IlceKodu,total_quantity,xfark,x_tavanfark,xipr,xipr_tavan,unit_price_change_day,is_price_fresh,...,kdv_oran,is_holiday,Arife,Tatil_Oncesi_Is_Gunu,Tatil_Ilk_Gun,Tatil_Son_Gun,Kac_Gun_Tatil,condition_group,Tatil_Turu,time_idx
0,2025-11-06,7,701,148.0,0.02,0.00,1.052380,1.528389,0,0,...,10.0,0,0,0,0,0,0,Yağmur,Normal_Gun,1039
1,2025-11-07,7,701,141.0,0.02,0.00,1.052380,1.528389,0,0,...,10.0,0,0,0,0,0,0,Yağmur,Normal_Gun,1040
2,2025-11-08,7,701,179.0,0.02,0.00,1.052380,1.528389,0,0,...,10.0,0,0,0,0,0,0,Bulutluluk,Normal_Gun,1041
3,2025-11-09,7,701,64.0,0.02,0.00,1.052380,1.528389,0,0,...,10.0,0,0,0,0,0,0,Bulutluluk,Normal_Gun,1042
4,2025-11-10,7,701,186.0,0.02,0.00,1.052380,1.528389,0,0,...,10.0,0,0,0,0,0,0,Bulutluluk,Normal_Gun,1043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1885,2026-02-14,7,765,40.0,0.04,0.09,0.887436,1.543346,2,1,...,10.0,0,0,0,0,0,0,Bulutluluk,Normal_Gun,1139
1886,2026-02-15,7,765,16.0,0.04,0.09,0.887436,1.543346,3,1,...,10.0,0,0,0,0,0,0,Yağmur,Normal_Gun,1140
1887,2026-02-16,7,765,63.0,0.04,0.09,0.887436,1.543346,4,1,...,10.0,0,0,0,0,0,0,Yağmur,Normal_Gun,1141
1888,2026-02-17,7,765,43.0,0.04,0.09,0.887436,1.543346,5,1,...,10.0,0,0,0,0,0,0,Yağmur,Normal_Gun,1142


In [8]:
import pickle
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet

# Model
model = TemporalFusionTransformer.load_from_checkpoint("checkpoints/fold1-best-epoch=27-val_loss=3.5235.ckpt")
model.eval()


TemporalFusionTransformer(
  	"attention_head_size":               4
  	"categorical_groups":                {}
  	"causal_attention":                  True
  	"dataset_parameters":                {'time_idx': 'time_idx', 'target': 'total_quantity', 'group_ids': ['IlceKodu'], 'weight': None, 'max_encoder_length': 90, 'min_encoder_length': 90, 'min_prediction_idx': np.int64(0), 'min_prediction_length': 14, 'max_prediction_length': 14, 'static_categoricals': ['IlceKodu', 'IlKodu'], 'static_reals': None, 'time_varying_known_categoricals': ['condition_group', 'Tatil_Turu'], 'time_varying_known_reals': ['xfark', 'x_tavanfark', 'xipr', 'xipr_tavan', 'unit_price_change_day', 'is_price_fresh', 'temp_c', 'month_sin', 'month_cos', 'day_sin', 'day_cos', 'kdv_oran', 'is_holiday', 'Arife', 'Tatil_Oncesi_Is_Gunu', 'Tatil_Ilk_Gun', 'Tatil_Son_Gun', 'Kac_Gun_Tatil'], 'time_varying_unknown_categoricals': None, 'time_varying_unknown_reals': ['total_quantity'], 'variable_groups': None, 'constant_fill_str

In [9]:
ham_df= pd.read_parquet("ham_data.parquet")

In [10]:
maaliyet_tl = ham_df["maaliyet_tl/tup"].iloc[-1]
tüpras_tl = ham_df["tüpras_tl/tup"].iloc[-1]
tavan_ayg = ham_df["unit_price_mode"].iloc[-1]
tavan_ipg = ham_df["fiyatTavan"].iloc[-1]

#### *Similasyon Hesaplaması*

In [11]:
# ============================================================
# REFERENCE DATASET'İ MODELDEN OLUŞTUR
# ============================================================
reference_dataset = TimeSeriesDataSet.from_parameters(
    model.dataset_parameters,
    base_df,
    predict=False,
    stop_randomization=True,
)
print("✅ reference_dataset oluşturuldu")
print(f"   Encoder len: {reference_dataset.max_encoder_length}")
print(f"   Decoder len: {reference_dataset.max_prediction_length}")

✅ reference_dataset oluşturuldu
   Encoder len: 90
   Decoder len: 14


In [12]:
model = model.to("cpu")
model.eval()

TemporalFusionTransformer(
  	"attention_head_size":               4
  	"categorical_groups":                {}
  	"causal_attention":                  True
  	"dataset_parameters":                {'time_idx': 'time_idx', 'target': 'total_quantity', 'group_ids': ['IlceKodu'], 'weight': None, 'max_encoder_length': 90, 'min_encoder_length': 90, 'min_prediction_idx': np.int64(0), 'min_prediction_length': 14, 'max_prediction_length': 14, 'static_categoricals': ['IlceKodu', 'IlKodu'], 'static_reals': None, 'time_varying_known_categoricals': ['condition_group', 'Tatil_Turu'], 'time_varying_known_reals': ['xfark', 'x_tavanfark', 'xipr', 'xipr_tavan', 'unit_price_change_day', 'is_price_fresh', 'temp_c', 'month_sin', 'month_cos', 'day_sin', 'day_cos', 'kdv_oran', 'is_holiday', 'Arife', 'Tatil_Oncesi_Is_Gunu', 'Tatil_Ilk_Gun', 'Tatil_Son_Gun', 'Kac_Gun_Tatil'], 'time_varying_unknown_categoricals': None, 'time_varying_unknown_reals': ['total_quantity'], 'variable_groups': None, 'constant_fill_str

In [13]:
# ============================================================
# 0. SETUP
# ============================================================
all_categorical_cols = ["IlceKodu", "IlKodu", "condition_group", "Tatil_Turu"]

float_cols = [
    "xfark", "x_tavanfark", "xipr", "xipr_tavan",
    "unit_price_change_day", "is_price_fresh",
    "temp_c",
    "month_sin", "month_cos", "day_sin", "day_cos",
    "kdv_oran",
    "is_holiday", "Arife", "Tatil_Oncesi_Is_Gunu",
    "Tatil_Ilk_Gun", "Tatil_Son_Gun", "Kac_Gun_Tatil",
    "total_quantity",
]

# Yeni model: encoder=90, decoder=14 → son 14 gün decoder
DECODER_LEN = 14
decoder_start_idx = base_df["time_idx"].max() - DECODER_LEN + 1
print(f"Decoder time_idx aralığı: {decoder_start_idx} → {base_df['time_idx'].max()}")

# ============================================================
# 1. SABİT TAVAN DEĞERLERİ (loop dışı, bir kez hesapla)
# ============================================================
x_tavan_val         = (tavan_ayg - maaliyet_tl) / tüpras_tl
xipr_tavan_val      = (tavan_ipg - maaliyet_tl) / tüpras_tl
x_tavanfark_val     = round(x_tavan_val - xipr_tavan_val, 2)   # tüm senaryolarda sabit
xipr_tavan_rounded  = round(xipr_tavan_val, 2)                 # tüm senaryolarda sabit

# ============================================================
# 2. SİMÜLASYON LOOP
# ============================================================
aygaz_range = range(900, 1400, 10)
ipr_range   = range(900, 1400, 10)

results = []

for ipr_val in tqdm(ipr_range, desc="İpragaz Senaryoları"):
    # ipr_val'a bağlı senaryolar (ayg loop'undan bağımsız) → loop dışına alınabilir
    xipr_scenario = round((ipr_val - maaliyet_tl) / tüpras_tl, 2)

    for ayg_val in aygaz_range:

        sim_df = base_df.copy()

        # --- Tip dönüşümleri ---
        for col in all_categorical_cols:
            if col in sim_df.columns:
                sim_df[col] = sim_df[col].astype(str)

        for col in float_cols:
            if col in sim_df.columns:
                sim_df[col] = sim_df[col].astype("float64")
            else:
                sim_df[col] = 0.0

        # --- Senaryo: aygaz tarafı ---
        x_scenario     = (ayg_val - maaliyet_tl) / tüpras_tl
        xfark_scenario = round(x_scenario - xipr_scenario, 2)   # eğitimle aynı: x - xipr

        # --- Sadece DECODER kısmını güncelle ---
        decoder_mask = sim_df["time_idx"] >= decoder_start_idx
        sim_df.loc[decoder_mask, "xipr"]        = xipr_scenario
        sim_df.loc[decoder_mask, "xipr_tavan"]  = xipr_tavan_rounded
        sim_df.loc[decoder_mask, "xfark"]       = xfark_scenario
        sim_df.loc[decoder_mask, "x_tavanfark"] = x_tavanfark_val

        # --- Dataset & dataloader ---
        sim_dataset = TimeSeriesDataSet.from_dataset(
            reference_dataset,
            sim_df,
            predict=True,
            stop_randomization=True,
        )
        sim_dataloader = sim_dataset.to_dataloader(
            train=False, batch_size=64, num_workers=0
        )

        # --- Tahmin ---
        model.eval()
        scenario_predictions = []

        with torch.no_grad():
            for x, y in sim_dataloader:
                output = model(x)

                if hasattr(output, "prediction"):
                    pred = output.prediction
                elif hasattr(output, "output"):
                    pred = output.output
                else:
                    pred = output

                # 7 quantile var, medyan = index 3 (q=0.5)
                if len(pred.shape) == 3:
                    pred = pred[:, :, 3]

                scenario_predictions.append(pred.cpu())

        preds         = torch.cat(scenario_predictions, dim=0).numpy()
        dataset_index = sim_dataset.index
        ilce_kodlari  = sim_df.iloc[dataset_index["index_start"].values]["IlceKodu"].values

        # --- Sonuçları topla ---
        for i, (_, idx_row) in enumerate(dataset_index.iterrows()):
            results.append({
                "aygaz_fiyat":      ayg_val,
                "ipr_fiyat":        ipr_val,
                "fiyat_fark":       ayg_val - ipr_val,
                "IlceKodu":         ilce_kodlari[i],
                "x_scenario":       x_scenario,
                "xipr_scenario":    xipr_scenario,
                "xfark":            xfark_scenario,
                "x_tavanfark":      x_tavanfark_val,
                "xipr_tavan":       xipr_tavan_rounded,
                "toplam_tahmin":    preds[i].sum(),
                "ortalama_tahmin":  preds[i].mean(),
                "min_tahmin":       preds[i].min(),
                "max_tahmin":       preds[i].max(),
            })

# ============================================================
# 3. SONUÇ
# ============================================================
results_df = pd.DataFrame(results)
results_df["IlKodu"] = il_nosu

print(f"Toplam senaryo: {results_df[['aygaz_fiyat','ipr_fiyat']].drop_duplicates().shape[0]}")
print(f"Toplam satır  : {len(results_df):,}")

Decoder time_idx aralığı: 1130 → 1143


İpragaz Senaryoları: 100%|██████████| 50/50 [03:49<00:00,  4.59s/it]

Toplam senaryo: 2500
Toplam satır  : 45,000


In [14]:
results_df.set_index

<bound method DataFrame.set_index of        aygaz_fiyat  ipr_fiyat  fiyat_fark IlceKodu  x_scenario  xipr_scenario  \
0              900        900           0      701    0.843708           0.84   
1              900        900           0      702    0.843708           0.84   
2              900        900           0      703    0.843708           0.84   
3              900        900           0      704    0.843708           0.84   
4              900        900           0      705    0.843708           0.84   
...            ...        ...         ...      ...         ...            ...   
44995         1390       1390           0      761    1.915028           1.92   
44996         1390       1390           0      762    1.915028           1.92   
44997         1390       1390           0      763    1.915028           1.92   
44998         1390       1390           0      764    1.915028           1.92   
44999         1390       1390           0      765    1.915028          

In [15]:
base_date = base_df["OrderDay"].min()

for i, (_, idx_row) in enumerate(dataset_index.iterrows()):
    for d in range(DECODER_LEN):
        results.append({
            "aygaz_fiyat":   ayg_val,
            "ipr_fiyat":     ipr_val,
            "fiyat_fark":    ayg_val - ipr_val,
            "IlceKodu":      ilce_kodlari[i],
            "OrderDay":      base_date + pd.Timedelta(days=int(decoder_start_idx + d)),
            "time_idx":      decoder_start_idx + d,
            "gun":           d + 1,             # 1, 2, ..., 14
            "tahmin":        preds[i][d],
            "x_scenario":    x_scenario,
            "xipr_scenario": xipr_scenario,
            "xfark":         xfark_scenario,
        })

#### *Grafik Datalarının Hazırlığı*

>***Geçmiş Satışların Eklenmesi***

In [16]:
df = ham_df[["fiilimalcıkıs", "distance_x", "total_quantity", "unit_price_mode", "discount_mode", "IlKodu", "IlceKodu"]].dropna(
    subset=["fiilimalcıkıs", "distance_x", "total_quantity"]
)
df = df.sort_values(["IlKodu", "IlceKodu", "fiilimalcıkıs"]).copy()

df["distance_x"] = df["distance_x"].round(6)

df["run_id"] = (
    df.groupby(["IlKodu", "IlceKodu"])["distance_x"]
    .transform(lambda x: x.ne(x.shift()).cumsum())
)

WINDOW = 7  

def forward_mean_same_run(s: pd.Series, window: int = WINDOW) -> pd.Series:
    """Run içindeki her pozisyon için i..i+window-1 arasının ortalaması (içerdiği kadar)."""
    arr = s.to_numpy(dtype=float)
    n = arr.size
    # Kümülatif toplam (prefix sum)
    pref = np.concatenate(([0.0], np.cumsum(arr)))
    idx = np.arange(n)
    j = np.minimum(idx + window, n)  # sağ sınır (açık aralık)
    length = j - idx                  # pencere uzunluğu (<= window)
    means = (pref[j] - pref[idx]) / length
    return pd.Series(means, index=s.index)

def forward_len_same_run(s: pd.Series, window: int = WINDOW) -> pd.Series:
    """Her pozisyon için kullanılan satır sayısı (kendi dâhil, en fazla window)."""
    n = len(s)
    idx = np.arange(n)
    j = np.minimum(idx + window, n)
    length = j - idx
    return pd.Series(length, index=s.index)

df["avg_next_same_dx_7"] = (
    df.groupby(["IlKodu", "IlceKodu", "run_id"])["total_quantity"]
    .transform(forward_mean_same_run)
)
df["rows_in_window"] = (
    df.groupby(["IlKodu", "IlceKodu", "run_id"])["total_quantity"]
    .transform(forward_len_same_run)
)


gecmis_df = pd.DataFrame({
    "IlKodu":        df["IlKodu"],
    "IlceKodu":      df["IlceKodu"],
    "start_date":    df["fiilimalcıkıs"],
    "distance_x":    df["distance_x"],
    "price":         df["unit_price_mode"] - df["discount_mode"],
    "avg_quantity":  df["avg_next_same_dx_7"],
    "rows_in_block": df["rows_in_window"],
})

gecmis_df["distance_x_tl"] = (
    gecmis_df["distance_x"] * ham_df["tüpras_tl/tup"].iloc[-1]
) + ham_df["unit_price_mode"].iloc[-1]

gecmis_df["avg_ton"] = (gecmis_df["avg_quantity"] / 12.0) * 1000.0


In [17]:
gecmis_df

,IlKodu,IlceKodu,start_date,distance_x,price,avg_quantity,rows_in_block,distance_x_tl,avg_ton
OrderDay,,,,,,,,,
2023-01-02,7,701,2023-01-02,-0.200493,296.0,85.142857,7,1168.298512,7095.238095
2023-01-03,7,701,2023-01-03,-0.200493,296.0,82.571429,7,1168.298512,6880.952381
2023-01-04,7,701,2023-01-04,-0.200493,296.0,82.000000,7,1168.298512,6833.333333
2023-01-05,7,701,2023-01-05,-0.200493,296.0,82.857143,7,1168.298512,6904.761905
2023-01-06,7,701,2023-01-06,-0.200493,306.0,80.142857,7,1168.298512,6678.571429
...,...,...,...,...,...,...,...,...,...
2026-02-14,7,765,2026-02-14,-0.699637,940.0,43.200000,5,940.000029,3600.000000
2026-02-15,7,765,2026-02-15,-0.699637,940.0,44.000000,4,940.000029,3666.666667
2026-02-16,7,765,2026-02-16,-0.699637,940.0,53.333333,3,940.000029,4444.444444


>***Geçmiş İpragaz Fiyatlarının Eklenmesi***

In [18]:
ham_df["gecmis_ipr"] = (
    ham_df["distance_xipr"] * ham_df["tüpras_tl/tup"].iloc[-1]
) + ham_df["fiyatTavan"].iloc[-1]

son_tarih  = ham_df["date"].max()
son_30_gun = ham_df[ham_df["date"] >= son_tarih - pd.Timedelta(days=30)]

In [19]:
ipr_frekans = (
    son_30_gun
    .groupby(["IlKodu", "IlceKodu", "gecmis_ipr"])
    .size()
    .reset_index(name="gun_sayisi")
)


>***Karlılık için Ağırlıklı Olarak Hesaplama***

In [20]:
bayiler_karlilik = pd.read_csv("tum_bayiler_karlılık.csv")
bayiler_karlilik["tupbasi_karlilik"] = bayiler_karlilik["ton_bası_karlılık"] / 1000 * 12

In [21]:
bayiler_karlilik["kunnr"]= bayiler_karlilik["kunnr"].astype(str)

In [22]:
fatura_saf = pd.read_parquet("fatura_saf.parquet")


kunnr_town = fatura_saf[["kunnr", "TOWN_ID", "PROVINCE_ID"]].drop_duplicates()
kunnr_town.columns = ["kunnr", "TOWN_ID", "PROVINCE_ID"]

# bayiler_karlilik ile KUNNR/kunnr üzerinden birleştir
bayiler_karlilik = bayiler_karlilik.merge(kunnr_town, on="kunnr", how="left")

bayiler_karlilik.head()

,kunnr,mart_miktar_ton,aygpay,hesnav,MRT 2026,fiil_nakliye_tl_ton,fob_tutar,ton_bası_karlılık,tupbasi_karlilik,TOWN_ID,PROVINCE_ID
0,1001056,5.22,35721.63,5342.49,-66154.0,1744.856079,NaN,26646.083845,319.753006,NaN,1.0
1,1001074,3.42,35721.63,5342.49,-32766.0,3860.651322,NaN,27622.766923,331.473203,12.0,1.0
2,1001128,2.28,35721.63,5342.48,-25509.0,2399.607343,NaN,27476.344762,329.716137,8.0,1.0
3,1001139,7.04,35721.63,5342.49,-106000.0,NaN,2.636486,26007.301818,312.087622,7.0,1.0
4,1001142,7.21,35721.62,5342.49,-85337.0,NaN,2.636486,29228.187670,350.738252,NaN,1.0


In [23]:
import numpy as np
import pandas as pd

# ============================================================
# 1. PROVINCE_ID = 7 FİLTRESİ
# ============================================================
df_il = bayiler_karlilik[bayiler_karlilik["PROVINCE_ID"] == 7].copy()
print(f"Toplam bayi (il=7): {len(df_il)}")
print(f"TOWN_ID NaN: {df_il['TOWN_ID'].isna().sum()} → düşürülecek")

df_il = df_il.dropna(subset=["TOWN_ID"]).copy()
df_il["TOWN_ID"] = df_il["TOWN_ID"].astype(int)

# ============================================================
# 2. TOWN BAZINDA HESAPLAMA
# ============================================================
def hesapla(grup):
    toplam_ton = grup["mart_miktar_ton"].sum()

    # --- tupbasi_karlilik: NaN olanları payda ve paydan çıkar (klasik weighted) ---
    mask_tup = grup["tupbasi_karlilik"].notna() & grup["mart_miktar_ton"].notna()
    if mask_tup.any() and grup.loc[mask_tup, "mart_miktar_ton"].sum() > 0:
        tupbasi_avg = (
            (grup.loc[mask_tup, "tupbasi_karlilik"] * grup.loc[mask_tup, "mart_miktar_ton"]).sum()
            / grup.loc[mask_tup, "mart_miktar_ton"].sum()
        )
    else:
        tupbasi_avg = np.nan

    # --- fob_tutar: PAY sadece fob dolu olanlardan, PAYDA tüm bayilerden ---
    fob_dolu = grup[grup["fob_tutar"].notna() & grup["mart_miktar_ton"].notna()]
    if len(fob_dolu) > 0 and toplam_ton > 0:
        fob_pay = (fob_dolu["fob_tutar"] * fob_dolu["mart_miktar_ton"]).sum()
        fob_avg = fob_pay / toplam_ton          # ← payda: TÜM bayilerin tonu
    else:
        fob_avg = np.nan

    return pd.Series({
        "bayi_sayisi":        len(grup),
        "fob_bayi_sayisi":    int(grup["fob_tutar"].notna().sum()),
        "toplam_mart_ton":    toplam_ton,
        "tupbasi_karlilik_w": tupbasi_avg,
        "fob_tutar_w":        fob_avg,
        "fob_kapsama_orani":  fob_dolu["mart_miktar_ton"].sum() / toplam_ton if toplam_ton > 0 else np.nan,
    })

town_ozet = (
    df_il.groupby("TOWN_ID", as_index=False)
         .apply(hesapla, include_groups=False)
         .reset_index(drop=True)
)

town_ozet = town_ozet.sort_values("toplam_mart_ton", ascending=False).reset_index(drop=True)
town_ozet

Toplam bayi (il=7): 50
TOWN_ID NaN: 0 → düşürülecek


,TOWN_ID,bayi_sayisi,fob_bayi_sayisi,toplam_mart_ton,tupbasi_karlilik_w,fob_tutar_w,fob_kapsama_orani
0,2037,9.0,9.0,117.18,329.279257,2.636486,1.0
1,132,8.0,8.0,104.92,355.943838,2.636486,1.0
2,2039,5.0,5.0,57.06,372.359380,2.636486,1.0
3,723,1.0,0.0,45.40,242.768368,NaN,0.0
4,134,2.0,0.0,39.54,281.119579,NaN,0.0
5,139,2.0,0.0,39.14,216.963499,NaN,0.0
6,1492,1.0,0.0,29.89,203.305001,NaN,0.0
7,1337,1.0,0.0,25.96,275.551686,NaN,0.0
8,2038,2.0,2.0,25.32,394.286654,2.636486,1.0
9,1451,2.0,0.0,21.56,302.199590,NaN,0.0


In [24]:
fatura_saf=fatura_saf[["tarihfatura","kunnr","hesmrj","baydgtpay","aygpay","PROVINCE_ID","TOWN_ID","adet"]]
fatura_saf["oran"] =  (fatura_saf["baydgtpay"] / fatura_saf["aygpay"]).round(2)
fatura_saf_sirali = fatura_saf.sort_values(by=['kunnr', 'tarihfatura'], ascending=[True, False])
son_durum_df = fatura_saf_sirali.drop_duplicates(subset=['kunnr'], keep='first').copy()

toplam_adet = fatura_saf.groupby('kunnr')['adet'].sum().reset_index()
toplam_adet.columns = ['kunnr', 'toplam_adet']

son_durum_df = son_durum_df.merge(toplam_adet, on='kunnr', how='left')

son_durum_df["oran"] = (son_durum_df["baydgtpay"] / son_durum_df["aygpay"]).round(2)
son_durum_df["hesmrj_tup_tl"] = (son_durum_df["hesmrj"] / 1000) * 12

son_durum_df=son_durum_df[son_durum_df["PROVINCE_ID"] == 7]

town_hacim = son_durum_df.groupby('TOWN_ID')['toplam_adet'].sum().reset_index()
town_hacim.columns = ['TOWN_ID', 'ilce_toplam_hacim']
analiz_df = pd.merge(son_durum_df, town_hacim, on='TOWN_ID')

analiz_df['bayi_pazar_payi'] = analiz_df['toplam_adet'] / analiz_df['ilce_toplam_hacim']

analiz_df['agirlikli_baydgt'] = analiz_df['baydgtpay'] * analiz_df['bayi_pazar_payi']
analiz_df['agirlikli_aygpay'] = analiz_df['aygpay'] * analiz_df['bayi_pazar_payi']
analiz_df['agirlikli_hesmrj'] = analiz_df['hesmrj'] * analiz_df['bayi_pazar_payi']

town_analiz = analiz_df.groupby('TOWN_ID').agg({
    'kunnr': 'count',
    'toplam_adet': 'sum',
    'agirlikli_baydgt': 'sum',
    'agirlikli_aygpay': 'sum',
    'agirlikli_hesmrj': 'sum'
}).reset_index()

town_analiz["ilce_oran_agirlikli"] = (town_analiz["agirlikli_baydgt"] / town_analiz["agirlikli_aygpay"]).round(2)
town_analiz["ilce_hesmrj_tup_tl"] = (town_analiz["agirlikli_hesmrj"] / 1000) * 12

ilce_df=pd.read_pickle("df_ilce_eslesme.pkl")


In [25]:
# ============================================================
# 1. FATURA → KUNNR BAZINDA SON DURUM
# ============================================================
fatura_saf = fatura_saf[
    ["tarihfatura", "kunnr", "hesmrj", "baydgtpay", "aygpay",
     "PROVINCE_ID", "TOWN_ID", "adet"]
]

fatura_saf_sirali = fatura_saf.sort_values(
    by=["kunnr", "tarihfatura"], ascending=[True, False]
)
son_durum_df = fatura_saf_sirali.drop_duplicates(subset=["kunnr"], keep="first").copy()

# Toplam adet (kunnr bazında)
toplam_adet = (
    fatura_saf.groupby("kunnr")["adet"].sum()
              .reset_index().rename(columns={"adet": "toplam_adet"})
)
son_durum_df = son_durum_df.merge(toplam_adet, on="kunnr", how="left")

# baydgtpay tüpe çevir
son_durum_df["baydgtpay_tup"] = (son_durum_df["baydgtpay"] / 1000) * 12

# Sadece il = 7
son_durum_df = son_durum_df[son_durum_df["PROVINCE_ID"] == 7].copy()
son_durum_df = son_durum_df.dropna(subset=["TOWN_ID"])
son_durum_df["TOWN_ID"] = son_durum_df["TOWN_ID"].astype(int)

# ============================================================
# 2. BAYİ TONAJ + İLÇE ÖZETİ İLE BİRLEŞTİR
# ============================================================
# kunnr bazında ton bilgisi (bayiler_karlilik'ten)
bayi_ton = bayiler_karlilik[["kunnr", "mart_miktar_ton"]].copy()

analiz_df = son_durum_df.merge(bayi_ton, on="kunnr", how="left")

# town_ozet'ten ilçe seviyesi bilgileri çek
analiz_df = analiz_df.merge(
    town_ozet[["TOWN_ID", "toplam_mart_ton", "tupbasi_karlilik_w",
               "fob_tutar_w", "fob_kapsama_orani"]],
    on="TOWN_ID", how="left"
)

# ============================================================
# 3. PAZAR PAYI (ton bazlı) + YENİ HESMRJ
# ============================================================
analiz_df["bayi_pazar_payi"] = (
    analiz_df["mart_miktar_ton"] / analiz_df["toplam_mart_ton"]
)

# Yeni hesmrj: bayinin tüp başı payı + ilçenin (Aygaz tarafı) tüp başı karı
analiz_df["hesmrj_yeni"] = (
    analiz_df["baydgtpay_tup"] + analiz_df["tupbasi_karlilik_w"]
)

# ============================================================
# 4. AĞIRLIKLI KOLONLAR (kunnr seviyesi, ton ağırlıklı)
# ============================================================
analiz_df["agirlikli_baydgt_tup"] = (
    analiz_df["baydgtpay_tup"] * analiz_df["bayi_pazar_payi"]
)
analiz_df["agirlikli_hesmrj"] = (
    analiz_df["hesmrj_yeni"] * analiz_df["bayi_pazar_payi"]
)

# ============================================================
# 5. İLÇE BAZINDA AGREGASYON
# ============================================================
town_analiz = (
    analiz_df.groupby("TOWN_ID")
    .agg(
        bayi_sayisi          = ("kunnr",                "count"),
        toplam_adet          = ("toplam_adet",          "sum"),
        toplam_mart_ton      = ("mart_miktar_ton",      "sum"),
        agirlikli_baydgt_tup = ("agirlikli_baydgt_tup", "sum"),
        agirlikli_hesmrj     = ("agirlikli_hesmrj",     "sum"),
        # town seviyesinden gelen, kunnr'lar arası aynı → first
        tupbasi_karlilik_w   = ("tupbasi_karlilik_w",   "first"),
        fob_tutar_w          = ("fob_tutar_w",          "first"),
        fob_kapsama_orani    = ("fob_kapsama_orani",    "first"),
    )
    .reset_index()
)

# Yeni oran: bayinin tüp başı payının ilçe Aygaz karına oranı
town_analiz["ilce_oran_agirlikli"] = (
    town_analiz["agirlikli_baydgt_tup"] / town_analiz["tupbasi_karlilik_w"]
).round(2)

# ============================================================
# 6. İLÇE EŞLEŞMESİ
# ============================================================
ilce_df = pd.read_pickle("df_ilce_eslesme.pkl")
town_analiz = town_analiz.merge(ilce_df, on="TOWN_ID", how="left")



In [26]:
town_analiz

,TOWN_ID,bayi_sayisi,toplam_adet,toplam_mart_ton,agirlikli_baydgt_tup,agirlikli_hesmrj,tupbasi_karlilik_w,fob_tutar_w,fob_kapsama_orani,ilce_oran_agirlikli,PROVINCE_ID,IlKodu,IlceKodu,Il,Ilce
0,132,1,38390.0,12.74,31.873280,75.094062,355.943838,2.636486,1.0,0.09,NaN,<NA>,<NA>,NaN,NaN
1,134,2,131093.0,39.54,262.491720,543.611299,281.119579,NaN,0.0,0.93,NaN,<NA>,<NA>,NaN,NaN
2,138,1,13092.0,1.56,235.218960,465.333295,230.114335,NaN,0.0,1.02,NaN,<NA>,<NA>,NaN,NaN
3,139,2,131996.0,39.14,262.491720,479.455219,216.963499,NaN,0.0,1.21,NaN,<NA>,<NA>,NaN,NaN
4,140,1,40520.0,5.94,235.218960,493.222902,258.003942,NaN,0.0,0.91,NaN,<NA>,<NA>,NaN,NaN
5,709,1,5854.0,3.30,235.218960,506.483448,271.264488,NaN,0.0,0.87,NaN,<NA>,<NA>,NaN,NaN
6,710,1,33906.0,0.00,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN
7,712,2,61584.0,10.18,524.983440,1346.673952,410.845256,2.636486,1.0,1.28,NaN,<NA>,<NA>,NaN,NaN
8,717,2,101281.0,10.99,262.491720,533.168504,270.676784,NaN,0.0,0.97,NaN,<NA>,<NA>,NaN,NaN
9,723,1,127902.0,45.40,262.491720,505.260088,242.768368,NaN,0.0,1.08,NaN,<NA>,<NA>,NaN,NaN


In [27]:
new_rows = pd.DataFrame([
    {"TOWN_ID": 1081, "PROVINCE_ID": 7, "IlKodu": 7, "IlceKodu": 765, "Il": "Antalya", "Ilce": "Demre"},
    {"TOWN_ID": 723,  "PROVINCE_ID": 7, "IlKodu": 7, "IlceKodu": 704, "Il": "Antalya", "Ilce": "Aksu"},
    {"TOWN_ID": 138,  "PROVINCE_ID": 7, "IlKodu": 7, "IlceKodu": 756, "Il": "Antalya", "Ilce": "Gündoğmuş"},
    {"TOWN_ID": 132,  "PROVINCE_ID": 7, "IlKodu": 7, "IlceKodu": 702, "Il": "Antalya", "Ilce": "Kepez"},
])

ilce_df = pd.concat([ilce_df, new_rows], ignore_index=True)

In [28]:
town_analiz["TOWN_ID"] = town_analiz["TOWN_ID"].astype(int)
ilce_df["TOWN_ID"] = ilce_df["TOWN_ID"].astype(int)
town_analiz=town_analiz.merge(ilce_df, left_on='TOWN_ID', right_on='TOWN_ID', how='left')

In [29]:
town_analiz

,TOWN_ID,bayi_sayisi,toplam_adet,toplam_mart_ton,agirlikli_baydgt_tup,agirlikli_hesmrj,tupbasi_karlilik_w,fob_tutar_w,fob_kapsama_orani,ilce_oran_agirlikli,PROVINCE_ID_x,IlKodu_x,IlceKodu_x,Il_x,Ilce_x,PROVINCE_ID_y,IlKodu_y,IlceKodu_y,Il_y,Ilce_y
0,132,1,38390.0,12.74,31.873280,75.094062,355.943838,2.636486,1.0,0.09,NaN,<NA>,<NA>,NaN,NaN,7.0,7,702,Antalya,Kepez
1,134,2,131093.0,39.54,262.491720,543.611299,281.119579,NaN,0.0,0.93,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN
2,138,1,13092.0,1.56,235.218960,465.333295,230.114335,NaN,0.0,1.02,NaN,<NA>,<NA>,NaN,NaN,7.0,7,756,Antalya,Gündoğmuş
3,139,2,131996.0,39.14,262.491720,479.455219,216.963499,NaN,0.0,1.21,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN
4,140,1,40520.0,5.94,235.218960,493.222902,258.003942,NaN,0.0,0.91,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN
5,709,1,5854.0,3.30,235.218960,506.483448,271.264488,NaN,0.0,0.87,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN
6,710,1,33906.0,0.00,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN
7,712,2,61584.0,10.18,524.983440,1346.673952,410.845256,2.636486,1.0,1.28,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN
8,717,2,101281.0,10.99,262.491720,533.168504,270.676784,NaN,0.0,0.97,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN
9,723,1,127902.0,45.40,262.491720,505.260088,242.768368,NaN,0.0,1.08,NaN,<NA>,<NA>,NaN,NaN,7.0,7,704,Antalya,Aksu


In [30]:
town_analiz["IlceKodu"] = town_analiz["IlceKodu_y"].astype(str)

In [31]:
results_df

,aygaz_fiyat,ipr_fiyat,fiyat_fark,IlceKodu,x_scenario,xipr_scenario,xfark,x_tavanfark,xipr_tavan,toplam_tahmin,ortalama_tahmin,min_tahmin,max_tahmin,IlKodu
0,900,900,0,701,0.843708,0.84,0.0,0.09,1.54,4038.706055,288.479004,133.781433,328.360168,7
1,900,900,0,702,0.843708,0.84,0.0,0.09,1.54,3859.451660,275.675110,154.191162,304.709106,7
2,900,900,0,703,0.843708,0.84,0.0,0.09,1.54,888.464417,63.461742,28.853832,73.738312,7
3,900,900,0,704,0.843708,0.84,0.0,0.09,1.54,1701.161865,121.511559,67.397255,138.154831,7
4,900,900,0,705,0.843708,0.84,0.0,0.09,1.54,1066.580566,76.184326,64.757416,79.570580,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,1390,1390,0,761,1.915028,1.92,-0.0,0.09,1.54,191.860046,13.704289,0.000006,21.029137,7
44996,1390,1390,0,762,1.915028,1.92,-0.0,0.09,1.54,547.916260,39.136875,9.171823,50.515022,7
44997,1390,1390,0,763,1.915028,1.92,-0.0,0.09,1.54,757.452026,54.103718,31.803715,61.495983,7
44998,1390,1390,0,764,1.915028,1.92,-0.0,0.09,1.54,47.663048,3.404503,0.210921,5.070901,7


In [32]:
results_df_kar=results_df.merge(town_analiz[["ilce_oran_agirlikli", "agirlikli_hesmrj","IlKodu_y","IlceKodu","toplam_adet", "agirlikli_baydgt_tup"]], left_on="IlceKodu", right_on="IlceKodu", how="left")
results_df_kar["toplam_kar"] = (results_df_kar["aygaz_fiyat"] - results_df_kar["agirlikli_hesmrj"]) * results_df_kar["ortalama_tahmin"]
results_df_kar["bayi_kar"]   = results_df_kar["agirlikli_baydgt_tup"] * results_df_kar["ortalama_tahmin"]
results_df_kar["aygaz_kar"]   = results_df_kar["toplam_kar"] - results_df_kar["bayi_kar"]

>***İl-İlçe hiyerarşisi***

In [33]:
il_ilce_map = (
    ham_df[["IlKodu", "Il", "IlceKodu", "Ilce"]]
    .drop_duplicates()
    .assign(
        IlKodu  =lambda x: x["IlKodu"].astype(str),
        IlceKodu=lambda x: x["IlceKodu"].astype(str)
    )
    .sort_values(["Il", "Ilce"])
)

il_listesi = il_ilce_map[["IlKodu", "Il"]].drop_duplicates()
il_options = [(row["Il"], row["IlKodu"]) for _, row in il_listesi.iterrows()]

ilk_il = il_options[0][1]
ilce_options_ilk = [
    (row["Ilce"], row["IlceKodu"])
    for _, row in il_ilce_map[il_ilce_map["IlKodu"] == ilk_il].iterrows()
]


In [34]:
ipr_secenekler = sorted(results_df_kar["ipr_fiyat"].unique())

>***Son Tarihler***

In [35]:
# ── Ön hazırlık — son fiyat hesapla ───────────────────────────
ham_df["IlceKodu"] = ham_df["IlceKodu"].astype(str)

# ── İlçe bazında son gelen fiyat + o günün miktarı ────────────
son_fiyat_ilce = (
    ham_df.sort_values('tarihFiyatGün')
    .groupby('IlceKodu')
    .last()[['dsc_price_tl/tup', 'total_quantity']]
    .reset_index()
    .rename(columns={'dsc_price_tl/tup': 'son_fiyat', 'total_quantity': 'son_miktar'})
)

# ── İl bazında son günün mod fiyatı + toplam miktarı ──────────
son_gun = ham_df['tarihFiyatGün'].max()
son_gun_df = ham_df[ham_df['tarihFiyatGün'] == son_gun]

son_fiyat_il = (
    son_gun_df.groupby('IlKodu')
    .agg(
        mod_fiyat=('dsc_price_tl/tup', lambda x: x.mode()[0]),
        son_miktar=('total_quantity', 'sum')
    )
    .reset_index()
)
son_fiyat_il["IlKodu"] = son_fiyat_il["IlKodu"].astype(str)

# ── Ön hazırlık — son ipragaz fiyatı ──────────────────────────
son_ipr_ilce = (
    ham_df.sort_values('tarihFiyatGün')
    .groupby('IlceKodu')['fiyatKampanya']
    .last()
    .reset_index()
    .rename(columns={'fiyatKampanya': 'son_ipr'})
)
son_ipr_ilce["IlceKodu"] = son_ipr_ilce["IlceKodu"].astype(str)

son_gun = ham_df['tarihFiyatGün'].max()
son_ipr_il = (
    ham_df[ham_df['tarihFiyatGün'] == son_gun]
    .groupby('IlKodu')['fiyatKampanya']
    .agg(lambda x: x.mode()[0])
    .reset_index()
    .rename(columns={'fiyatKampanya': 'son_ipr'})
)
son_ipr_il["IlKodu"] = son_ipr_il["IlKodu"].astype(str)

#### Öneri Motoru

In [36]:
df

,fiilimalcıkıs,distance_x,total_quantity,unit_price_mode,discount_mode,IlKodu,IlceKodu,run_id,avg_next_same_dx_7,rows_in_window
OrderDay,,,,,,,,,,
2023-01-02,2023-01-02,-0.200493,108.0,331.0,35.0,7,701,1,85.142857,7
2023-01-03,2023-01-03,-0.200493,85.0,331.0,35.0,7,701,1,82.571429,7
2023-01-04,2023-01-04,-0.200493,74.0,331.0,35.0,7,701,1,82.000000,7
2023-01-05,2023-01-05,-0.200493,96.0,331.0,35.0,7,701,1,82.857143,7
2023-01-06,2023-01-06,-0.200493,82.0,341.0,35.0,7,701,1,80.142857,7
...,...,...,...,...,...,...,...,...,...,...
2026-02-14,2026-02-14,-0.699637,40.0,1260.0,320.0,7,765,71,43.200000,5
2026-02-15,2026-02-15,-0.699637,16.0,1260.0,320.0,7,765,71,44.000000,4
2026-02-16,2026-02-16,-0.699637,63.0,1260.0,320.0,7,765,71,53.333333,3


In [37]:



# ── Widget'lar ─────────────────────────────────────────────────
il_dropdown = widgets.Dropdown(
    options=il_options, value=ilk_il,
    description='İl Seç:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)
ilce_dropdown = widgets.Dropdown(
    options=ilce_options_ilk, value=ilce_options_ilk[0][1],
    description='İlçe Seç:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)
ipr_dropdown = widgets.Dropdown(
    options=[(f"{f:.2f} ₺", f) for f in ipr_secenekler],
    value=ipr_secenekler[len(ipr_secenekler)//2],
    description='İpragaz Fiyatı:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)
il_butonu = widgets.Button(
    description='📊 İl Görünümü', button_style='info',
    layout=widgets.Layout(width='160px')
)
ilce_butonu = widgets.Button(
    description='🔍 İlçe Görünümü', button_style='warning',
    layout=widgets.Layout(width='160px')
)
agirlik_slider = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05,
    description='Karlılık Dengesi:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px'),
    readout_format='.2f'
)
agirlik_label = widgets.HTML(
    "<span style='font-size:12px; color:#666'>← Karlılık odaklı &nbsp;&nbsp;&nbsp;&nbsp; Miktar odaklı →</span>"
)
slider_box = widgets.VBox([agirlik_slider, agirlik_label])
output = widgets.Output()


# ── Yardımcı fonksiyonlar ──────────────────────────────────────
def _gecmis_scatter(gecmis, ad='Gerçek Noktalar'):
    return go.Scatter(
        x=gecmis['distance_x_tl'], y=gecmis['avg_quantity'],
        mode='markers', marker=dict(color='gray', size=7, opacity=0.35),
        name=ad,
        hovertemplate='Fiyat: %{x:.2f} ₺<br>Satış: %{y:.2f}<extra></extra>'
    )

def _son30_scatter(gecmis, ad='Son 30 Gün'):
    son_tarih = gecmis['start_date'].max()
    son_30    = gecmis[gecmis['start_date'] >= son_tarih - pd.Timedelta(days=30)]
    return go.Scatter(
        x=son_30['distance_x_tl'], y=son_30['avg_quantity'],
        mode='markers',
        marker=dict(color='rgba(30, 100, 255, 0.30)', size=9),
        name=ad,
        hovertemplate='Fiyat: %{x:.2f} ₺<br>Satış: %{y:.2f}<extra></extra>'
    )

def _kar_traces(x, toplam, bayi, aygaz):
    return [
        go.Scatter(x=x, y=toplam, yaxis='y2', mode='lines',
            name='Toplam Karlılık',
            line=dict(color='goldenrod', width=2, dash='dot'),
            hovertemplate='Aygaz: %{x:.2f} ₺<br>Toplam Kar: %{y:,.0f} ₺<extra></extra>'),
        go.Scatter(x=x, y=bayi, yaxis='y2', mode='lines',
            name='Bayi Karlılığı',
            line=dict(color='tomato', width=2, dash='dash'),
            hovertemplate='Aygaz: %{x:.2f} ₺<br>Bayi Kar: %{y:,.0f} ₺<extra></extra>'),
        go.Scatter(x=x, y=aygaz, yaxis='y2', mode='lines',
            name='Aygaz Karlılığı',
            line=dict(color='steelblue', width=2, dash='dashdot'),
            hovertemplate='Aygaz: %{x:.2f} ₺<br>Aygaz Kar: %{y:,.0f} ₺<extra></extra>'),
    ]

def _shared_layout(title, yaxis_title, ipr_fiyat, gun_label=''):
    return dict(
        title=f'{title} | İpragaz: {ipr_fiyat:.2f} ₺{gun_label}',
        xaxis_title='Aygaz Fiyatı (₺)',
        yaxis_title=yaxis_title,
        yaxis2=dict(
            title=dict(text='Karlılık (₺)', font=dict(color='goldenrod')),  # ← DEĞİŞTİ
            overlaying='y', side='right',
            showgrid=False,
            tickfont=dict(color='goldenrod'),
        ),
        yaxis3=dict(
            overlaying='y', side='right',
            showgrid=False, showticklabels=False,
            showline=False, zeroline=False,
            range=[-1, 1],
        ),
        template='plotly_white', hovermode='x unified', height=580,
        legend=dict(orientation='h', y=-0.20)
    )

def _ipr_vline(fig, ipr_fiyat):
    fig.add_vline(x=ipr_fiyat, line_width=2, line_dash='dash', line_color='crimson',
        annotation_text=f'İpragaz: {ipr_fiyat:.2f} ₺',
        annotation_position='top left',
        annotation_font=dict(color='crimson', size=12))

def _yildiz_trace(x, y, label):
    return go.Scatter(
        x=[x], y=[y], mode='markers',
        marker=dict(symbol='star', size=20, color='limegreen',
                    line=dict(color='darkgreen', width=1.5)),
        name=label,
        hovertemplate=f'{label}<br>Fiyat: {x:.2f} ₺<br>Miktar: {y:.2f}<extra></extra>',
        showlegend=True
    )

def _ipr_kutu_trace(ipr_df, yaxis='y3'):
    if len(ipr_df) == 0:
        return None
    maks    = ipr_df['gun_sayisi'].max()
    renkler = [
        f'rgba(128, 0, 128, {0.10 + 0.75 * (g / maks):.2f})'
        for g in ipr_df['gun_sayisi']
    ]
    return go.Scatter(
        x=ipr_df['gecmis_ipr'],
        y=[0] * len(ipr_df),
        yaxis=yaxis, mode='markers',
        marker=dict(symbol='square', size=18, color=renkler,
                    line=dict(color='purple', width=1)),
        name='İpragaz Geçmiş Fiyatlar',
        hovertemplate='İpragaz: %{x:.2f} ₺<br>%{customdata} gün<extra></extra>',
        customdata=ipr_df['gun_sayisi'],
        showlegend=True
    )

def optimum_bul(df, agirlik_miktar=0.5):
    m = df['ortalama_tahmin']
    k = df['aygaz_kar']
    m_norm = (m - m.min()) / (m.max() - m.min()) if m.max() > m.min() else m * 0 + 1
    k_norm = (k - k.min()) / (k.max() - k.min()) if k.max() > k.min() else k * 0 + 1
    skor   = agirlik_miktar * m_norm + (1 - agirlik_miktar) * k_norm
    idx    = skor.idxmax()
    return df.loc[idx, 'aygaz_fiyat'], df.loc[idx, 'ortalama_tahmin'], df.loc[idx, 'aygaz_kar']

def _oneri_kutusu(mevcut_fiyat, opt_fiyat, mevcut_miktar, opt_miktar, mevcut_kar, opt_kar):
    fark        = opt_fiyat - mevcut_fiyat
    miktar_fark = opt_miktar - mevcut_miktar
    kar_fark    = opt_kar - mevcut_kar

    if abs(fark) < 0.01:
        yon_text = '✅ Mevcut fiyat zaten optimum noktada!'
        renk = '#1565c0'
    elif fark < 0:
        yon_text = f'📉 {abs(fark):.2f} ₺ indirim önerisi'
        renk = '#c62828'
    else:
        yon_text = f'📈 {fark:.2f} ₺ artış önerisi'
        renk = '#2e7d32'

    miktar_text = f"+{miktar_fark:.1f}" if miktar_fark >= 0 else f"{miktar_fark:.1f}"
    kar_text    = f"+{kar_fark:,.0f}" if kar_fark >= 0 else f"{kar_fark:,.0f}"

    html = f"""
    <div style="
        background: linear-gradient(135deg, #f8f9fa, #e8f4fd);
        border-left: 5px solid {renk};
        border-radius: 8px;
        padding: 12px 18px;
        margin: 8px 0;
        font-family: Arial, sans-serif;
    ">
        <div style="font-size:15px; font-weight:bold; color:{renk}; margin-bottom:8px;">
            {yon_text}
        </div>
        <table style="border-collapse:collapse; width:100%; font-size:13px;">
            <tr style="color:#666; font-size:12px;">
                <td style="padding:3px 15px 3px 0"></td>
                <td style="padding:3px 15px 3px 0">Mevcut</td>
                <td style="padding:3px 15px 3px 0">Optimum</td>
                <td style="padding:3px 0">Fark</td>
            </tr>
            <tr>
                <td style="color:#666; padding:4px 15px 4px 0">Fiyat</td>
                <td style="font-weight:bold">{mevcut_fiyat:.2f} ₺</td>
                <td style="font-weight:bold; color:{renk}">{opt_fiyat:.2f} ₺</td>
                <td style="font-weight:bold; color:{renk}">{fark:+.2f} ₺</td>
            </tr>
            <tr>
                <td style="color:#666; padding:4px 15px 4px 0">Günlük Miktar</td>
                <td style="font-weight:bold">{mevcut_miktar:.1f} adet</td>
                <td style="font-weight:bold">{opt_miktar:.1f} adet</td>
                <td style="font-weight:bold; color:{'#2e7d32' if miktar_fark>=0 else '#c62828'}">{miktar_text} adet</td>
            </tr>
            <tr>
                <td style="color:#666; padding:4px 15px 4px 0">Aygaz Karlılık</td>
                <td style="font-weight:bold">{mevcut_kar:,.0f} ₺</td>
                <td style="font-weight:bold">{opt_kar:,.0f} ₺</td>
                <td style="font-weight:bold; color:{'#2e7d32' if kar_fark>=0 else '#c62828'}">{kar_text} ₺</td>
            </tr>
        </table>
    </div>
    """
    return widgets.HTML(html)


# ── İlçe grafik ────────────────────────────────────────────────
def ilce_grafigi(secilen_ilce, secilen_ipr):
    df = results_df_kar[
        (results_df_kar['IlceKodu'] == secilen_ilce) &
        (results_df_kar['ipr_fiyat'] == secilen_ipr)
    ].sort_values('aygaz_fiyat')

    gecmis   = gecmis_df[gecmis_df['IlceKodu'] == secilen_ilce]
    ilce_adi = il_ilce_map[il_ilce_map['IlceKodu'] == secilen_ilce]['Ilce'].iloc[0]

    gun_sayisi = ipr_frekans[
        (ipr_frekans['IlceKodu'].astype(str) == secilen_ilce) &
        (ipr_frekans['gecmis_ipr'] == secilen_ipr)
    ]['gun_sayisi'].sum()
    gun_label = f' — son 30 günde {gun_sayisi} gün' if gun_sayisi > 0 else ''

    fig = go.Figure()

    # Geçmiş noktalar
    fig.add_trace(_gecmis_scatter(gecmis))
    fig.add_trace(_son30_scatter(gecmis))

    # Tahmin çizgisi
    fig.add_trace(go.Scatter(
        x=df['aygaz_fiyat'], y=df['ortalama_tahmin'],
        mode='lines+markers', name='Ortalama Satış Tahmini',
        line=dict(color='royalblue', width=2.5), marker=dict(size=6),
        hovertemplate='Aygaz: %{x:.2f} ₺<br>Tahmin: %{y:.2f} adet<extra></extra>'
    ))

    # Karlılık çizgileri
    for trace in _kar_traces(df['aygaz_fiyat'], df['toplam_kar'], df['bayi_kar'], df['aygaz_kar']):
        fig.add_trace(trace)

    # İpragaz kutuları
    ipr_ilce = ipr_frekans[ipr_frekans['IlceKodu'].astype(str) == secilen_ilce]
    kutu = _ipr_kutu_trace(ipr_ilce)
    if kutu:
        fig.add_trace(kutu)

    # Yeşil yıldız
    ilce_son = son_fiyat_ilce[son_fiyat_ilce['IlceKodu'] == secilen_ilce]
    if not ilce_son.empty:
        son_f = ilce_son['son_fiyat'].iloc[0]
        son_m = ilce_son['son_miktar'].iloc[0]
        fig.add_trace(_yildiz_trace(son_f, son_m, f'Son Fiyat ({son_f:.2f} ₺)'))

    # Dikey çizgiler
    _ipr_vline(fig, secilen_ipr)

    fig.add_vline(x=tavan_ayg, line_width=2, line_dash='dot', line_color='darkorange',
        annotation_text=f'Aygaz Tavan: {tavan_ayg:.2f} ₺',
        annotation_position='top right',
        annotation_font=dict(color='darkorange', size=12))

    ipr_son = son_ipr_ilce[son_ipr_ilce['IlceKodu'] == secilen_ilce]
    if not ipr_son.empty:
        s = ipr_son['son_ipr'].iloc[0]
        fig.add_vline(x=s, line_width=2, line_dash='dot', line_color='purple',
            annotation_text=f'Son İpragaz: {s:.2f} ₺',
            annotation_position='bottom right',
            annotation_font=dict(color='purple', size=12))

    # Optimum çizgisi
    opt_fiyat, opt_miktar, opt_kar = optimum_bul(df, agirlik_slider.value)
    fig.add_vline(x=opt_fiyat, line_width=2.5, line_dash='solid', line_color='gold',
        annotation_text=f'⭐ Optimum: {opt_fiyat:.2f} ₺',
        annotation_position='top left',
        annotation_font=dict(color='goldenrod', size=13))

    fig.update_layout(**_shared_layout(
        f'Talep & Karlılık — {ilce_adi}',
        'Ortalama Günlük Satış (adet)',
        secilen_ipr, gun_label
    ))
    fig.show()

    # Öneri kutusu
    if not df.empty:
        mevcut_f = ilce_son['son_fiyat'].iloc[0]
        mevcut_m = (df.loc[df["aygaz_fiyat"] == ilce_son["son_fiyat"].iloc[0], "ortalama_tahmin"].iloc[0]).round(0) 
        mevcut_k = df.loc[df['aygaz_fiyat'].sub(mevcut_f).abs().idxmin(), 'aygaz_kar'] \
                   if len(df) > 0 else 0
        display(_oneri_kutusu(mevcut_f, opt_fiyat, mevcut_m, opt_miktar, mevcut_k, opt_kar))


# ── İl grafik ──────────────────────────────────────────────────
def il_grafigi(secilen_il, secilen_ipr):
    il_ilceler = il_ilce_map[il_ilce_map['IlKodu'] == secilen_il]['IlceKodu'].tolist()
    il_adi     = il_ilce_map[il_ilce_map['IlKodu'] == secilen_il]['Il'].iloc[0]

    df_il_raw = results_df_kar[
        (results_df_kar['IlceKodu'].isin(il_ilceler)) &
        (results_df_kar['ipr_fiyat'] == secilen_ipr)
    ].copy()

    # İlçe ağırlıkları
    ilce_agirlik = df_il_raw.groupby('IlceKodu')['toplam_adet'].first().fillna(0)
    toplam_hacim = ilce_agirlik.sum()
    ilce_agirlik = (ilce_agirlik / toplam_hacim) if toplam_hacim > 0 else ilce_agirlik
    df_il_raw    = df_il_raw.join(ilce_agirlik.rename('agirlik'), on='IlceKodu')

    df_il = (
        df_il_raw.groupby('aygaz_fiyat')
        .apply(lambda g: pd.Series({
            'ortalama_tahmin': g['ortalama_tahmin'].sum(),
            'toplam_kar':      (g['toplam_kar'] * g['agirlik']).sum() * len(il_ilceler),
            'bayi_kar':        (g['bayi_kar']   * g['agirlik']).sum() * len(il_ilceler),
            'aygaz_kar':       (g['aygaz_kar']  * g['agirlik']).sum() * len(il_ilceler),
        }))
        .reset_index()
        .sort_values('aygaz_fiyat')
    )

    gecmis_il = (
        gecmis_df[gecmis_df['IlceKodu'].isin(il_ilceler)]
        .groupby('start_date')
        .agg(avg_quantity=('avg_quantity', 'sum'),
             distance_x_tl=('distance_x_tl', lambda x: x.mode()[0]))
        .reset_index()
    )
    son_tarih    = gecmis_df['start_date'].max()
    gecmis_il_30 = (
        gecmis_df[
            (gecmis_df['IlceKodu'].isin(il_ilceler)) &
            (gecmis_df['start_date'] >= son_tarih - pd.Timedelta(days=30))
        ]
        .groupby('start_date')
        .agg(avg_quantity=('avg_quantity', 'sum'),
             distance_x_tl=('distance_x_tl', lambda x: x.mode()[0]))
        .reset_index()
    )

    gun_toplam = ipr_frekans[
        (ipr_frekans['IlceKodu'].astype(str).isin(il_ilceler)) &
        (ipr_frekans['gecmis_ipr'] == secilen_ipr)
    ]['gun_sayisi'].sum()
    gun_label = f' — son 30 günde {gun_toplam} gün' if gun_toplam > 0 else ''

    fig = go.Figure()

    # Geçmiş noktalar
    fig.add_trace(_gecmis_scatter(gecmis_il, 'Gerçek Noktalar (İl)'))
    fig.add_trace(go.Scatter(
        x=gecmis_il_30['distance_x_tl'], y=gecmis_il_30['avg_quantity'],
        mode='markers',
        marker=dict(color='rgba(30, 100, 255, 0.30)', size=9),
        name='Son 30 Gün (İl)',
        hovertemplate='Fiyat: %{x:.2f} ₺<br>Satış: %{y:.2f}<extra></extra>'
    ))

    # Tahmin çizgisi
    fig.add_trace(go.Scatter(
        x=df_il['aygaz_fiyat'], y=df_il['ortalama_tahmin'],
        mode='lines+markers', name='İl Toplam Satış Tahmini',
        line=dict(color='seagreen', width=2.5), marker=dict(size=6),
        hovertemplate='Aygaz: %{x:.2f} ₺<br>Toplam Tahmin: %{y:.2f} adet<extra></extra>'
    ))

    # Karlılık çizgileri
    for trace in _kar_traces(df_il['aygaz_fiyat'], df_il['toplam_kar'], df_il['bayi_kar'], df_il['aygaz_kar']):
        fig.add_trace(trace)

    # İpragaz kutuları
    ipr_il = (
        ipr_frekans[ipr_frekans['IlceKodu'].astype(str).isin(il_ilceler)]
        .groupby('gecmis_ipr')['gun_sayisi']
        .sum()
        .reset_index()
    )
    kutu = _ipr_kutu_trace(ipr_il)
    if kutu:
        fig.add_trace(kutu)

    # Yeşil yıldız
    il_mod = son_fiyat_il[son_fiyat_il['IlKodu'] == secilen_il]
    if not il_mod.empty:
        mod_f = il_mod['mod_fiyat'].iloc[0]
        mod_m = il_mod['son_miktar'].iloc[0]
        fig.add_trace(_yildiz_trace(mod_f, mod_m, f'Son Gün ({mod_f:.2f} ₺)'))

    # Dikey çizgiler
    _ipr_vline(fig, secilen_ipr)

    fig.add_vline(x=tavan_ayg, line_width=2, line_dash='dot', line_color='darkorange',
        annotation_text=f'Aygaz Tavan: {tavan_ayg:.2f} ₺',
        annotation_position='top right',
        annotation_font=dict(color='darkorange', size=12))

    ipr_son_il = son_ipr_il[son_ipr_il['IlKodu'] == secilen_il]
    if not ipr_son_il.empty:
        s = ipr_son_il['son_ipr'].iloc[0]
        fig.add_vline(x=s, line_width=2, line_dash='dot', line_color='purple',
            annotation_text=f'Son İpragaz: {s:.2f} ₺',
            annotation_position='bottom right',
            annotation_font=dict(color='purple', size=12))

    # Optimum çizgisi
    opt_fiyat_il, opt_miktar_il, opt_kar_il = optimum_bul(df_il, agirlik_slider.value)
    fig.add_vline(x=opt_fiyat_il, line_width=2.5, line_dash='solid', line_color='gold',
        annotation_text=f'⭐ Optimum: {opt_fiyat_il:.2f} ₺',
        annotation_position='top left',
        annotation_font=dict(color='goldenrod', size=13))

    fig.update_layout(**_shared_layout(
        f'Talep & Karlılık — {il_adi} İl Toplamı',
        'Toplam Günlük Satış (adet)',
        secilen_ipr, gun_label
    ))
    fig.show()

    # Öneri kutusu
    if not il_mod.empty:
        mevcut_f = il_mod['mod_fiyat'].iloc[0]
        mevcut_m = (df_il.loc[df_il["aygaz_fiyat"] == il_mod['mod_fiyat'].iloc[0], "ortalama_tahmin"].iloc[0]).round(0) 
        mevcut_k = df_il.loc[df_il['aygaz_fiyat'].sub(mevcut_f).abs().idxmin(), 'aygaz_kar'] \
                   if len(df_il) > 0 else 0
        display(_oneri_kutusu(mevcut_f, opt_fiyat_il, mevcut_m, opt_miktar_il, mevcut_k, opt_kar_il))


# ── Mod & olaylar ──────────────────────────────────────────────
mod = {'aktif': 'il'}

def grafigi_goster():
    with output:
        output.clear_output(wait=True)
        if mod['aktif'] == 'il':
            il_grafigi(il_dropdown.value, ipr_dropdown.value)
        else:
            ilce_grafigi(ilce_dropdown.value, ipr_dropdown.value)

def il_goster(_):
    mod['aktif'] = 'il'
    kontrol_satiri.children = [il_dropdown, ipr_dropdown, ilce_butonu]
    grafigi_goster()

def ilce_goster(_):
    mod['aktif'] = 'ilce'
    kontrol_satiri.children = [il_dropdown, ilce_dropdown, ipr_dropdown, il_butonu]
    grafigi_goster()

def il_degisti(change):
    if change['name'] == 'value':
        yeni_ilceler = [
            (row["Ilce"], row["IlceKodu"])
            for _, row in il_ilce_map[il_ilce_map["IlKodu"] == change['new']].iterrows()
        ]
        ilce_dropdown.unobserve_all()
        ilce_dropdown.options = yeni_ilceler
        ilce_dropdown.value   = yeni_ilceler[0][1]
        ilce_dropdown.observe(lambda c: grafigi_goster() if c['name'] == 'value' else None)
        grafigi_goster()

il_dropdown.observe(il_degisti, names='value')
ilce_dropdown.observe(lambda c: grafigi_goster() if c['name'] == 'value' else None)
ipr_dropdown.observe(lambda c: grafigi_goster() if c['name'] == 'value' else None)
agirlik_slider.observe(lambda c: grafigi_goster() if c['name'] == 'value' else None)
il_butonu.on_click(il_goster)
ilce_butonu.on_click(ilce_goster)

# ── İlk görünüm ────────────────────────────────────────────────
kontrol_satiri = widgets.HBox([il_dropdown, ipr_dropdown, ilce_butonu])
display(slider_box, kontrol_satiri, output)
with output:
    il_grafigi(il_dropdown.value, ipr_dropdown.value)

Output()

In [40]:
karlılık = pd.read_csv("tum_bayiler_karlılık.csv")

In [41]:
karlılık

,kunnr,mart_miktar_ton,aygpay,hesnav,MRT 2026,fiil_nakliye_tl_ton,fob_tutar,ton_bası_karlılık
0,1001056,5.22,35721.63,5342.49,-66154.0,1744.856079,NaN,26646.083845
1,1001074,3.42,35721.63,5342.49,-32766.0,3860.651322,NaN,27622.766923
2,1001128,2.28,35721.63,5342.48,-25509.0,2399.607343,NaN,27476.344762
3,1001139,7.04,35721.63,5342.49,-106000.0,NaN,2.636486,26007.301818
4,1001142,7.21,35721.62,5342.49,-85337.0,NaN,2.636486,29228.187670
...,...,...,...,...,...,...,...,...
858,1441010,9.64,35809.56,5309.49,NaN,1515.144004,4.593498,NaN
859,1446006,1.80,35094.18,5344.99,-21600.0,3788.552467,NaN,24650.617533
860,1450004,2.16,34913.14,4276.13,-12960.0,6034.294142,NaN,27154.975858
861,1467011,8.58,35738.22,5380.82,-113402.0,3919.766760,NaN,23982.256923
